# CIDC25 — Exploratory Data Analysis

Goal: **understand the data before picking a model**. We answer six questions:

1. Dtype & intensity range per file.
2. Noise model: is `Var(pixel)` affine in `Mean(pixel)`? (Poisson-Gaussian check)
3. How different are noise levels across train (`*1` vs `*2`) and val (`F1/F2/F3`)?
4. Temporal autocorrelation — do transients really decay over many frames?
5. Spatial stats — neuron size, sparsity.
6. `F0` (clean) vs `F1/F2/F3`: is it purely additive noise? Any scale/offset?

Everything heavy lives in `workspace/src/cidc/`. This notebook only orchestrates.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src' / 'cidc').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('workspace root:', ROOT)

import numpy as np
import matplotlib.pyplot as plt
from cidc import (
    load_stack, stack_info, mean_var_per_pixel,
    estimate_poisson_gaussian, temporal_autocorr,
)

DATA = ROOT / 'data'
TRAIN = sorted((DATA / 'train').glob('*.tif'))
VAL   = sorted((DATA / 'val').glob('*.tif'))
print('train:', [p.name for p in TRAIN])
print('val  :', [p.name for p in VAL])

## 1. Basic info per stack

Dtype, shape, intensity range. Tells us normalisation strategy.

In [ ]:
for p in TRAIN + VAL:
    info = stack_info(p)
    print(f'{p.name:8s}  shape={info.shape}  dtype={info.dtype}  '
          f'min={info.min:.1f}  mean={info.mean:.1f}  max={info.max:.1f}')

## 2. Noise model — Poisson-Gaussian fit

For each stack, compute per-pixel (mean, var) along time and fit `Var = gain*Mean + read_var`. A clean linear fit with `R^2 > ~0.9` on the low-intensity band confirms Poisson-Gaussian. The slope ≈ detector gain; the intercept ≈ read-noise variance.

**Caveat:** active (firing) pixels inflate variance far above the noise line. We mitigate with `trim`; for a publication-grade fit you'd restrict to pixels whose mean is below the population median (mostly background).

In [ ]:
def fit_and_plot(path, ax, max_pixels=150_000):
    arr = load_stack(path)
    m, v = mean_var_per_pixel(arr, max_pixels=max_pixels)
    # Restrict fit to background-ish pixels (below median mean).
    bg = m < np.median(m)
    fit = estimate_poisson_gaussian(m[bg], v[bg])
    ax.scatter(m, v, s=1, alpha=0.15)
    xs = np.linspace(m.min(), m.max(), 50)
    ax.plot(xs, fit.gain * xs + fit.read_var, 'r-', lw=1.5,
            label=f'gain={fit.gain:.3f}  read_var={fit.read_var:.1f}  R²={fit.r2:.2f}')
    ax.set_title(path.name); ax.set_xlabel('mean'); ax.set_ylabel('var')
    ax.legend(fontsize=8)
    return fit

stacks = TRAIN + VAL
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fits = {}
for p, ax in zip(stacks, axes.ravel()):
    fits[p.name] = fit_and_plot(p, ax)
plt.tight_layout(); plt.show()

print('\nGain / read_var per file:')
for k, f in fits.items():
    print(f'  {k:8s}  gain={f.gain:+.3f}  read_var={f.read_var:+.1f}  R2={f.r2:.2f}')

## 3. Noise-level comparison

Training files ending in `1` should be less noisy than those ending in `2`. Validation has `F1, F2, F3` of increasing noise. Compare fitted gains: higher gain ⇒ more shot noise per count ⇒ noisier.

In [ ]:
names = list(fits)
gains = [fits[n].gain for n in names]
plt.figure(figsize=(8, 3))
plt.bar(names, gains)
plt.ylabel('fitted gain'); plt.title('Gain per stack (proxy for noise level)')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 4. Temporal autocorrelation

If the autocorrelation at lag 1–30 is high, neighbouring frames share signal and differ mostly in noise → temporal denoisers (DeepInterpolation / DeepCAD) will work. If it falls to zero at lag 1, only spatial denoising will help.

Calcium transients typically have τ ≈ 200–500 ms; at 30 Hz that's ~10–15 frames.

In [ ]:
plt.figure(figsize=(7, 4))
for p in stacks:
    acf = temporal_autocorr(load_stack(p), max_lag=60, max_pixels=1500)
    plt.plot(acf, label=p.name)
plt.axhline(0, c='k', lw=0.5)
plt.xlabel('lag (frames)'); plt.ylabel('ACF'); plt.title('Temporal autocorrelation')
plt.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()

## 5. Spatial snapshot

One frame + its temporal mean. The temporal mean is a near-zero-cost denoiser (it is the MLE denoise under static signal + independent noise); it sets a **lower bound** on how good a learned denoiser must be to be useful.

In [ ]:
sample = VAL[0] if VAL else TRAIN[0]
arr = load_stack(sample)
frame = np.asarray(arr[arr.shape[0] // 2])
tmean = np.asarray(arr[::10]).mean(axis=0)  # subsample time for speed
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img, title in zip(axes, [frame, tmean], [f'{sample.name}: single frame', f'{sample.name}: temporal mean (T/10)']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Clean vs noisy (validation only)

We have `F0` (clean) and `F1/F2/F3` (noisy versions of the same sample). Check:

- `Fk = a*F0 + b + noise`? Scatter `F0` vs `Fk` → should be close to `y=x`.
- Histogram of residuals `Fk - F0` should look Poisson-Gaussian.
- `var(Fk - F0)` vs `F0` → should lie on the same Poisson-Gaussian line as §2.

In [ ]:
f0_path = DATA / 'val' / 'F0.tif'
if not f0_path.exists():
    print('F0.tif not downloaded yet; skipping')
else:
    F0 = load_stack(f0_path)
    for name in ['F1.tif', 'F2.tif', 'F3.tif']:
        p = DATA / 'val' / name
        if not p.exists():
            print(f'{name} missing, skipping'); continue
        Fk = load_stack(p)
        # Use a small spatial subset and a few frames for speed.
        t_idx = np.linspace(0, F0.shape[0]-1, 60, dtype=int)
        a = np.asarray(F0[t_idx, :128, :128], dtype=np.float64)
        b = np.asarray(Fk[t_idx, :128, :128], dtype=np.float64)
        resid = b - a
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
        axes[0].scatter(a.ravel()[::50], b.ravel()[::50], s=1, alpha=0.2)
        lo, hi = a.min(), a.max()
        axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1)
        axes[0].set_title(f'F0 vs {name}'); axes[0].set_xlabel('F0'); axes[0].set_ylabel(name)
        axes[1].hist(resid.ravel(), bins=120)
        axes[1].set_title(f'{name} - F0  (mean={resid.mean():.2f}, std={resid.std():.2f})')
        # var of residual vs intensity → should match Poisson-Gaussian line
        bins = np.linspace(a.min(), a.max(), 40)
        which = np.digitize(a.ravel(), bins)
        mean_bin, var_bin = [], []
        for i in range(1, len(bins)):
            m = which == i
            if m.sum() > 100:
                mean_bin.append(a.ravel()[m].mean())
                var_bin.append(resid.ravel()[m].var())
        axes[2].plot(mean_bin, var_bin, 'o-')
        axes[2].set_xlabel('F0 intensity'); axes[2].set_ylabel(f'Var({name} - F0)')
        axes[2].set_title('Noise-vs-intensity (residual)')
        plt.tight_layout(); plt.show()

## What to conclude

After running all cells, write down in `docs/concepts.md`:

- Final dtype & normalisation choice.
- Estimated `(gain, read_var)` per noise level.
- Rough temporal decay (frames for ACF to fall to 0.5).
- Whether `F1/F2/F3` are consistent with the same forward model as train.

These numbers drive every downstream decision: loss function, augmentation range (for Task 2 OOD noise), Anscombe vs raw input, window size for temporal models.